# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haadjunejo/Flyrank-ML-01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# --- Path bootstrap: makes this notebook work whether it's opened inside your
# cloned repo (normal case) or uploaded standalone (Colab "Upload notebook") ---
import os

def _find_repo_root():
    here = os.getcwd()
    for path in [here, os.path.join(here, "Flyrank-ML-01")]:
        if os.path.exists(os.path.join(path, "data", "raw", "content_refresh_anonymized.csv")):
            return path
        candidate = os.path.join(path, "work", "notebooks")
        if os.path.exists(os.path.join(candidate, "..", "..", "data", "raw", "content_refresh_anonymized.csv")):
            return candidate
    return None

root = _find_repo_root()
if root is None:
    if not os.path.exists("Flyrank-ML-01"):
        !git clone -q https://github.com/haadjunejo/Flyrank-ML-01.git
    os.chdir("Flyrank-ML-01/work/notebooks")
elif root != os.getcwd():
    os.chdir(root)

print("Working directory:", os.getcwd())
assert os.path.exists("../../data/raw/content_refresh_anonymized.csv"), "Data file still not found -- check repo clone"
print("Data file found OK.")


Working directory: /content/Flyrank-ML-01/work/notebooks
Data file found OK.


## 1. My rule and its reason codes

Before coding a rule, I check the signals it would lean on. Two checks below, one of them
flag-linked (CTR-vs-position, the signal behind FlyRank's real `low_ctr_visible_page` /
CTR-fix logic), each with a bucket table + n + a one-word verdict.

### Signal check 1 -- staleness (`days_since_last_update` / `freshness_tier`), flag-linked to `stale_visible_page`

The refresh flag's assumption: staler content performs worse / is more likely to be declining.
I check it by bucketing on `freshness_tier` and looking at the decline rate per bucket.

In [2]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

signal1 = (df.groupby("freshness_tier")
             .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
             .sort_values("decline_rate"))
print(signal1)


                    n  decline_rate
freshness_tier                     
181+              174      0.471264
0-30            20480      0.511377
31-90             175      0.588571
91-180           9171      0.611057


**Verdict: OPPOSITE.** The relationship isn't just weak, it runs the wrong way: the stalest
pages (`181+`, n=174) have the LOWEST decline rate (47.1%), while the medium tier (`91-180`,
n=9,171) has the HIGHEST (61.1%) -- freshest pages (`0-30`) sit in between at 51.1%. If
staleness alone predicted decline, `181+` should be the worst bucket, not the best. My read:
pages that have been stale the longest already bottomed out and plateaued, while pages in the
`91-180` window are still actively losing ground. **This is a real, explained negative, and it
changes my rule** -- I will not gate or heavily weight my score on raw staleness.

### Signal check 2 -- CTR vs position tier, flag-linked to `low_ctr_visible_page` / CTR-fix logic

The CTR-fix flag's assumption: pages at better positions should earn higher CTR, so a page
that has a good position but low CTR is genuinely underperforming (not just naturally low-CTR
for its rank). I check by bucketing on `position_tier` and looking at mean CTR per bucket.

In [3]:
order = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
signal2 = (df.groupby("position_tier")
             .agg(n=("content_id", "size"), mean_ctr=("ctr", "mean"))
             .reindex(order).dropna(how="all"))
print(signal2)


                     n  mean_ctr
position_tier                   
top_3           2321.0  1.483611
page_1         11814.0  0.652467
striking        7304.0  0.323239
page_3_5        7242.0  0.222484
deep            1319.0  0.150212


**Verdict: CONFIRMED.** CTR drops cleanly and monotonically as position gets worse:
`top_3` (1.48%, n=2,321) -> `page_1` (0.65%, n=11,814) -> `striking` (0.32%, n=7,304) ->
`page_3_5` (0.22%, n=7,242) -> `deep` (0.15%, n=1,319). This holds up exactly as the flag
assumes, so I build my rule's primary signal around it: compare each page's own CTR against
the **median CTR for its own position tier** (not a single global threshold), and flag the gap.

### The rule, in plain words

A page is worth reviewing first if it has **real search demand** (enough impressions to
matter) and its **click-through rate sits below what other pages at its own position tier
typically earn** -- meaning it's leaving clicks on the table it should already be capturing,
given where it already ranks. I deliberately do NOT weight raw staleness, since signal check 1
showed that signal doesn't hold up. I also deliberately do NOT use `trend_direction` or
`trend_pct` anywhere in the score or reason code -- those are the label source (per the Week 3
contract), and using them here would make my baseline circular against the very metric
(`is_declining_label`, precision@K) I'll compare it to later.

**Score:** `demand_ok * (0.70 * ctr_gap_percentile + 0.30 * demand_percentile)`, where
`ctr_gap = max(0, tier_median_ctr - page_ctr)` and `demand_ok = impressions_90d >= 300`.

**Reason code (exactly one per row, priority order):**
1. `ctr_underperformer_visible` -- demand ok, real CTR gap, and `impressions_90d >= 500`
2. `ctr_underperformer_moderate` -- demand ok, real CTR gap, lower volume
3. `visible_review_candidate` -- demand ok, but CTR already at/above its tier's median
4. `low_priority_monitor` -- demand gate fails (`impressions_90d < 300`)

**Action label:** `ctr_underperformer_*` -> `review_title_meta_snippet`;
`visible_review_candidate` -> `monitor_or_expand`; `low_priority_monitor` -> `monitor`.

## 2. Build the ranked queue (writes the CSV)

In [4]:
import numpy as np
from pathlib import Path

def pct_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

expected_ctr_by_tier = df.groupby("position_tier")["ctr"].transform("median")
df["ctr_gap"] = (expected_ctr_by_tier - df["ctr"]).clip(lower=0)
df["demand_ok"] = (df["impressions_90d"] >= 300).astype(int)
df["ctr_gap_score"] = pct_rank(df["ctr_gap"])
df["demand_score"] = pct_rank(np.log1p(df["impressions_90d"]))

df["baseline_action_score"] = (
    df["demand_ok"] * (0.70 * df["ctr_gap_score"] + 0.30 * df["demand_score"])
).round(4)

def reason_code(row):
    if row["demand_ok"] == 0:
        return "low_priority_monitor"
    if row["ctr_gap"] > 0 and row["impressions_90d"] >= 500:
        return "ctr_underperformer_visible"
    if row["ctr_gap"] > 0:
        return "ctr_underperformer_moderate"
    return "visible_review_candidate"

def action_label(reason):
    if reason.startswith("ctr_underperformer"):
        return "review_title_meta_snippet"
    if reason == "visible_review_candidate":
        return "monitor_or_expand"
    return "monitor"

df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = df["reason_code"].apply(action_label)
df["rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

out_cols = ["content_id", "client_id", "rank", "baseline_action_score", "reason_code", "action",
            "impressions_90d", "clicks_90d", "avg_position", "position_tier", "ctr", "ctr_gap",
            "content_type", "word_count", "days_since_last_update", "freshness_tier",
            "trend_direction", "is_declining_label"]
out = df[out_cols].sort_values("rank")

print(out["reason_code"].value_counts())

output_path = Path("../outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(output_path, index=False)
print(f"\nWrote {len(out):,} ranked rows to {output_path}")

print(f"Base rate (share declining, whole dataset): {out['is_declining_label'].mean():.3f}")
print(f"Top-50 declining rate under this rule: {out.head(50)['is_declining_label'].mean():.3f}")


reason_code
visible_review_candidate       12735
low_priority_monitor           11248
ctr_underperformer_visible      4973
ctr_underperformer_moderate     1044
Name: count, dtype: int64

Wrote 30,000 ranked rows to ../outputs/baseline_action_score.csv
Base rate (share declining, whole dataset): 0.542
Top-50 declining rate under this rule: 0.720


## 3. Top-10 review

For each of the top 10: the action, why it's there, and what would make it wrong.

In [5]:
top10 = out.head(10).copy()
for _, row in top10.iterrows():
    print(f"#{row['rank']} {row['content_id']} | score={row['baseline_action_score']:.3f} | "
          f"{row['reason_code']} -> {row['action']}")
    print(f"    impressions_90d={row['impressions_90d']:,.0f}, clicks_90d={row['clicks_90d']}, "
          f"avg_position={row['avg_position']}, position_tier={row['position_tier']}, "
          f"ctr={row['ctr']}, ctr_gap={row['ctr_gap']:.3f}")
    print(f"    word_count={row['word_count']}, days_since_update={row['days_since_last_update']}, "
          f"freshness_tier={row['freshness_tier']}\n")


#1 content_c8e9d6ab9013 | score=0.953 | ctr_underperformer_visible -> review_title_meta_snippet
    impressions_90d=208,678, clicks_90d=0, avg_position=9.7, position_tier=page_1, ctr=0.0, ctr_gap=0.160
    word_count=nan, days_since_update=104, freshness_tier=91-180

#2 content_f986bd514b6e | score=0.938 | ctr_underperformer_visible -> review_title_meta_snippet
    impressions_90d=22,456, clicks_90d=1, avg_position=6.6, position_tier=page_1, ctr=0.0, ctr_gap=0.160
    word_count=3803.0, days_since_update=20, freshness_tier=0-30

#3 content_825a9788af8d | score=0.931 | ctr_underperformer_visible -> review_title_meta_snippet
    impressions_90d=16,786, clicks_90d=0, avg_position=5.6, position_tier=page_1, ctr=0.0, ctr_gap=0.160
    word_count=nan, days_since_update=104, freshness_tier=91-180

#4 content_8ba781dafa55 | score=0.930 | ctr_underperformer_visible -> review_title_meta_snippet
    impressions_90d=16,156, clicks_90d=0, avg_position=9.0, position_tier=page_1, ctr=0.0, ctr_gap=0.1

**Row-by-row review** (content_id shortened to first 12 chars):

1. `content_c8e9d6a` -- **review_title_meta_snippet.** Huge demand (208,678 impressions),
   page_1 position (9.7), but `ctr=0.0` -- flagged as the single biggest CTR gap in the whole
   dataset. *What would make it wrong:* `clicks_90d` is also exactly 0, not just rounded-down
   -- with this much volume, a true 0-click page is more likely a tracking/attribution problem
   (e.g. clicks landing on a different tracked URL) than a fixable title/snippet issue.
2. `content_f986bd5` -- same reason code, 22,456 impressions, position 6.6, `ctr=0.0`.
   *What would make it wrong:* word_count=3,803 (a substantial page) with literally zero
   clicks at position 6.6 is unusual enough that I'd sanity-check the GSC property mapping
   before trusting this as a content problem.
3. `content_825a978` -- 16,786 impressions, position 5.6, `ctr=0.0`. *What would make it
   wrong:* `word_count` is missing (NaN) here -- if the content record itself is incomplete,
   the "review title/snippet" action may be premature until the record is fixed.
4. `content_8ba781d` -- 16,156 impressions, position 9.0, `ctr=0.0`. *What would make it
   wrong:* stale for 104 days AND zero CTR is a plausible real fix candidate, but the identical
   `ctr_gap=0.160` across rows #1-4 (all snapped to the same tier median) means the rule can't
   yet tell which of these zero-CTR pages is worse than another -- they're tied on my score's
   main driver.
5. `content_5d5653c` -- 15,101 impressions, position 5.7, `ctr=0.0`, but `trend_direction=stable`
   (not declining). *What would make it wrong:* if this page's traffic isn't actually
   declining, "urgent CTR fix" may overstate the priority relative to pages that are visibly
   losing ground too.
6. `content_847a841` -- 14,519 impressions, position 7.4, `ctr=0.0`, also `trend_direction=stable`.
   *What would make it wrong:* same caveat as #5 -- stable trend, so the case for urgency rests
   on CTR alone.
7. `content_c82bc0c` -- 13,676 impressions, position 4.3 (very strong), `ctr=0.0`. *What would
   make it wrong:* position 4.3 with zero clicks is the most surprising row in the top 10 --
   worth checking first whether this URL is even the one actually ranking (redirects, canonical
   mismatches) before assuming the snippet itself is the problem.
8. `content_9983d31` -- 7,737 impressions, position 5.5, `ctr=0.0`, updated 7 days ago. *What
   would make it wrong:* freshly updated already -- if it was just edited and still shows
   `ctr=0.0`, the CTR-fix action may already be moot or the data hasn't caught up yet.
9. `content_d3aaf7d` -- 7,732 impressions, position 8.3, `ctr=0.0`, word_count missing. *What
   would make it wrong:* same incomplete-record caveat as #3.
10. `content_453722` -- 140,079 impressions, position 7.6, `ctr=0.01` (not exactly zero this
    time). *What would make it wrong:* this is the most defensible pick in the top 10 -- real,
    nonzero-but-tiny CTR at huge volume and strong position is exactly the pattern the rule was
    built to catch, with none of the zero-click oddities above.

## 4. Weak picks + leakage check

**The pattern that stands out:** 9 of my top 10 picks have `ctr` exactly `0.0` -- not just low,
literally zero, and their `clicks_90d` confirms it (also 0, not a rounding artifact). Checking
how common this is across the whole declining-CTR pool below.

In [6]:
zero_ctr_high_impr = df[(df["ctr"] == 0) & (df["impressions_90d"] >= 1000)]
print(f"Rows with ctr==0.0 AND impressions_90d>=1000: {len(zero_ctr_high_impr):,}")
print(zero_ctr_high_impr["clicks_90d"].describe())

print("\nThis is a real, systematic pattern, not a fluke of my top 10 -- "
      f"{len(zero_ctr_high_impr):,} pages combine real search volume with a literal zero "
      "click count. My rule currently can't distinguish 'genuinely fixable CTR problem' from "
      "'possible tracking/attribution issue' -- that's this baseline's clearest weakness, and "
      "exactly the kind of judgment a human reviewer needs to apply before acting on the "
      "top of this queue.")

print("\nLeakage check: confirming trend_direction / trend_pct were never used as scoring "
      "inputs anywhere above (only reported alongside the output for context).")
score_inputs = ["demand_ok", "ctr_gap_score", "demand_score"]
print(f"Score built only from: {score_inputs} -- no trend_direction / trend_pct / "
      "is_declining_label used in scoring, reason codes, or actions.")


Rows with ctr==0.0 AND impressions_90d>=1000: 1,157
count    1157.000000
mean        0.009507
std         0.127850
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         3.000000
Name: clicks_90d, dtype: float64

This is a real, systematic pattern, not a fluke of my top 10 -- 1,157 pages combine real search volume with a literal zero click count. My rule currently can't distinguish 'genuinely fixable CTR problem' from 'possible tracking/attribution issue' -- that's this baseline's clearest weakness, and exactly the kind of judgment a human reviewer needs to apply before acting on the top of this queue.

Leakage check: confirming trend_direction / trend_pct were never used as scoring inputs anywhere above (only reported alongside the output for context).
Score built only from: ['demand_ok', 'ctr_gap_score', 'demand_score'] -- no trend_direction / trend_pct / is_declining_label used in scoring, reason codes, or actions.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.